In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

!wget "https://drive.google.com/uc?export=download&id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU" -O healthyfime.csv
df = pd.read_csv("healthyfime.csv")

df.replace({"M": 0, "F": 1}, inplace=True)
classes = list(df['class'].unique())
mapping_dict = {ch: i for i, ch in enumerate(sorted(classes, reverse=True))}
df['class'].replace(mapping_dict, inplace=True)

X, y = df.iloc[:, :-1], df.iloc[:, -1]
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.1, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

--2026-10-06 15:38:02--  https://drive.google.com/uc?export=download&id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU
Resolving drive.google.com (drive.google.com)... 173.194.44.101, 173.194.44.139, 173.194.44.100, ...
Connecting to drive.google.com (drive.google.com)|173.194.44.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU&export=download [following]
--2026-10-06 15:38:02--  https://drive.usercontent.google.com/download?id=1GVhrh2rH6hUunV4Tf7lQoSFsqov9aPjU&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.98.132, 2607:f8b0:400c:c1a::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.98.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 761835 (744K) [application/octet-stream]
Saving to: ‘healthyfime.csv’

healthyfime.csv     100%[===================>] 743.98K  --.-KB/s

/tmp/ipykernel_685/409210731.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"M": 0, "F": 1}, inplace=True)
/tmp/ipykernel_685/409210731.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['class'].replace(mapping_dict, inplace=True)
/tmp/ipykernel_685/409210731.py:24: FutureWar

In [ ]:
# Rebuild + retrain the baseline model from Part 1
baseline_model = nn.Sequential(
    nn.Linear(11, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 4)
    # nn.Softmax(dim=1) -- lead to double softmax if used with CE loss
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)

n_epochs = 500
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(n_epochs):
    baseline_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(baseline_model(xb), yb)
        loss.backward()
        optimizer.step()

    baseline_model.eval()
    with torch.no_grad():
        tr_preds, va_preds = baseline_model(X_train_t), baseline_model(X_val_t)
        history["train_loss"].append(criterion(tr_preds, y_train_t).item())
        history["val_loss"].append(criterion(va_preds, y_val_t).item())
        history["train_acc"].append((tr_preds.argmax(1) == y_train_t).float().mean().item())
        history["val_acc"].append((va_preds.argmax(1) == y_val_t).float().mean().item())

print("Baseline trained. Final val accuracy:", round(history["val_acc"][-1], 4))

Baseline trained. Final val accuracy: 0.7529


In [ ]:
class WorkoutClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(11, 32)
    self.relu1 = nn.ReLU()
    self.fc2 = nn.Linear(32, 16)
    self.relu2 = nn.ReLU()
    self.fc3 = nn.Linear(16, 4)

  def forward(self, x):
    x = self.relu1(self.fc1(x))
    x = self.relu2(self.fc2(x))
    x = self.fc3(x)
    return x

model_class_version = WorkoutClassifier()
print(model_class_version)

WorkoutClassifier(
  (fc1): Linear(in_features=11, out_features=32, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=16, out_features=4, bias=True)
)


In [ ]:
class MultiBranchModel(nn.Module):
  def __init__(self):
    super().__init__()

    #Branch 1
    self.branch1_fc1 = nn.Linear(11, 16)
    self.branch1_fc2 = nn.Linear(16, 8)

    # Branch 2
    self.branch2_fc1 = nn.Linear(5, 4)

    # Output Layer
    self.out1 = nn.Linear(8, 4) # Branch 1
    self.out2 = nn.Linear(4, 1) # Branch 2

  def forward(self, x1, x2):
    h1 = torch.relu(self.branch1_fc1(x1)) # Branch 1
    h1 = torch.relu(self.branch1_fc2(h1)) # Branch 1

    h2 = torch.tanh(self.branch2_fc1(x2)) # Branch 2

    out1 = self.out1(h1)   # raw logits for classification
    out2 = self.out2(h2)   # raw value for regression
    return out1, out2

multi_model = MultiBranchModel()
dummy_x1 = torch.randn(8, 11)
dummy_x2 = torch.randn(8, 5)
logits, score = multi_model(dummy_x1, dummy_x2)
print(logits.shape, score.shape)

torch.Size([8, 4]) torch.Size([8, 1])


In [ ]:
logits

tensor([[ 0.0487,  0.2734, -0.4378,  0.1344],
        [ 0.0100,  0.2174, -0.3372,  0.1146],
        [ 0.2129,  0.2952, -0.2668,  0.2451],
        [ 0.0878,  0.4004, -0.2195,  0.2473],
        [ 0.0578,  0.2871, -0.3040,  0.1962],
        [ 0.2369,  0.2926, -0.2795,  0.2834],
        [-0.0534,  0.3202, -0.3046,  0.0701],
        [-0.1146,  0.2899, -0.3169,  0.0587]], grad_fn=<AddmmBackward0>)

In [ ]:
score

tensor([[-0.1118],
        [ 0.0206],
        [-0.0773],
        [-0.3084],
        [-0.4013],
        [-0.1911],
        [-0.3422],
        [-0.3088]], grad_fn=<AddmmBackward0>)

Model Prediction and Evaluation in Pytorch

In [ ]:
def evaluate_model(model, X_t, y_t, criterion):
    model.eval() # 1. Switch layers to evaluation mode
    with torch.no_grad(): # 2. Turn off gradient tracking to save memory/speed up inference
        preds = model(X_t)
        loss = criterion(preds, y_t).item()
        acc = (preds.argmax(1) == y_t).float().mean().item()
        print(acc)
    return loss, acc

for name, Xs, ys in [("Train", X_train_t, y_train_t), ("Val", X_val_t, y_val_t), ("Test", X_test_t, y_test_t)]:
    loss, acc = evaluate_model(baseline_model, Xs, ys, criterion)
    print(f"{name:5s} — loss: {loss:.2f}  accuracy: {acc:.2f}")

0.7912787199020386
Train — loss: 0.52  accuracy: 0.79
0.7529021501541138
Val   — loss: 0.59  accuracy: 0.75
0.7619403004646301
Test  — loss: 0.62  accuracy: 0.76


In [ ]:
baseline_model.eval()
with torch.no_grad():
    logits = baseline_model(X_test_t)
    probs = torch.softmax(logits, dim=1)
    pred_class = probs.argmax(dim=1)

print("Sum of probabilities per row (should be ~1):", probs.sum(dim=1)[:5])
print("Predicted classes:", pred_class[:10])

acc_score = accuracy_score(y_test_t.numpy(), pred_class.numpy())
print("Cross-checked test accuracy:", acc_score)

Sum of probabilities per row (should be ~1): tensor([1., 1., 1., 1., 1.])
Predicted classes: tensor([0, 3, 2, 1, 0, 3, 2, 3, 1, 3])
Cross-checked test accuracy: 0.7619402985074627


In [ ]:
# Quiz 4 -
# a. Training degrades — CrossEntropyLoss already applies softmax internally, so the model's raw scores get squashed twice, distorting the gradient signal.

In [ ]:
class MultiOutputReviewModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Linear(11, 16)
        self.class_head = nn.Linear(16, 4)     # classification output
        self.score_head = nn.Linear(16, 1)     # regression output

    def forward(self, x):
        h = torch.relu(self.shared(x))
        class_out = self.class_head(h)
        score_out = torch.relu(self.score_head(h))   # <-- worth a second look
        return class_out, score_out

Model Monitoring and Controlling

In [ ]:
model_es = nn.Sequential(
    nn.Linear(11, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 4)
)
optimizer_es = torch.optim.Adam(model_es.parameters(), lr=0.001)

best_val_loss = float("inf")
patience = 20
epochs_without_improvement = 0
n_epochs_es = 500

for epoch in range(n_epochs_es):
    model_es.train()
    for xb, yb in train_loader:
        optimizer_es.zero_grad()
        loss = criterion(model_es(xb), yb)
        loss.backward()
        optimizer_es.step()

    model_es.eval()
    with torch.no_grad():
        val_loss = criterion(model_es(X_val_t), y_val_t).item()

    # Model Checkpoint
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      epochs_without_improvement = 0
      torch.save(model_es.state_dict(), "best_model.pt")
      print(model_es.state_dict())
    else:
      epochs_without_improvement += 1

    # Early Stopping
    if epochs_without_improvement >= patience:
      print(f"  Early stopping at epoch {epoch+1}.")
      break



Streaming output truncated to the last 5000 lines.
          1.5473e-01],
        [-4.1157e-01, -4.3694e-01,  9.9602e-02,  1.1717e-01, -8.4095e-02,
          8.2448e-02, -2.0123e-03, -8.0023e-02,  6.6591e-02, -8.3622e-01,
         -2.2356e-01],
        [ 1.1556e-01, -5.0109e-01,  2.1520e-01, -3.6942e-01, -6.7195e-01,
         -5.9755e-02,  1.3354e-01,  3.4365e-01, -3.2661e-01, -1.6990e-01,
          1.9057e-02],
        [ 2.0939e-01, -1.3699e-02,  1.1004e-01, -3.0385e-01,  8.7757e-02,
         -4.2793e-02,  1.4653e-01, -2.7892e-01,  1.1888e-01,  2.8747e-01,
          1.5387e-01],
        [ 4.0935e-01, -1.5291e-01, -1.3837e-02, -1.5604e-01,  1.8704e-01,
         -6.1328e-02,  1.4492e-01,  3.3255e-01,  7.8307e-02,  4.2001e-01,
          4.1711e-01],
        [-1.0950e-01, -2.8317e-01, -3.2715e-01,  2.8412e-01,  1.0056e-01,
          5.7722e-02,  1.7305e-02, -1.3693e-01,  2.6973e-01,  1.9893e-01,
          1.0301e-01],
        [-1.0760e-01, -2.4877e-01,  1.1810e-02,  1.9379e-01, -1.1411e-0